# Overview of Automated Evals

In [1]:
import warnings
warnings.filterwarnings('ignore')

## Load API tokens for our 3rd party APIs.

In [2]:
from utils import get_circle_api_key
cci_api_key = get_circle_api_key()

In [3]:
from utils import get_gh_api_key
gh_api_key = get_gh_api_key()

In [4]:
from utils import get_openai_api_key
openai_api_key = get_openai_api_key()

## Set up our github branch

In [5]:
from utils import get_repo_name
course_repo = get_repo_name()
course_repo

'holaholu/llmops-course'

In [6]:
from utils import get_branch
course_branch = get_branch()
course_branch

'dl-cci-best-selling-puck-80'

## The sample application: AI-powered quiz generator
We are going to build a AI powered quiz generator.

Create the dataset for the quizz.

In [7]:
human_template  = "{question}"

In [8]:
quiz_bank = """1. Subject: Leonardo DaVinci
   Categories: Art, Science
   Facts:
    - Painted the Mona Lisa
    - Studied zoology, anatomy, geology, optics
    - Designed a flying machine
  
2. Subject: Paris
   Categories: Art, Geography
   Facts:
    - Location of the Louvre, the museum where the Mona Lisa is displayed
    - Capital of France
    - Most populous city in France
    - Where Radium and Polonium were discovered by scientists Marie and Pierre Curie

3. Subject: Telescopes
   Category: Science
   Facts:
    - Device to observe different objects
    - The first refracting telescopes were invented in the Netherlands in the 17th Century
    - The James Webb space telescope is the largest telescope in space. It uses a gold-berillyum mirror

4. Subject: Starry Night
   Category: Art
   Facts:
    - Painted by Vincent van Gogh in 1889
    - Captures the east-facing view of van Gogh's room in Saint-Rémy-de-Provence

5. Subject: Physics
   Category: Science
   Facts:
    - The sun doesn't change color during sunset.
    - Water slows the speed of light
    - The Eiffel Tower in Paris is taller in the summer than the winter due to expansion of the metal."""

Build the prompt template.

In [9]:
delimiter = "####"

prompt_template = f"""
Follow these steps to generate a customized quiz for the user.
The question will be delimited with four hashtags i.e {delimiter}

The user will provide a category that they want to create a quiz for. Any questions included in the quiz
should only refer to the category.

Step 1:#### First decide whether the user is asking about a subject that is in the following quiz bank:
- Geography: Paris, Rome (historical), Mediterranean climate
- Science: Photosynthesis, Newton's Laws, Quantum Mechanics  
- Art: Mona Lisa, Starry Night, Impressionism

Here is the quiz bank

{quiz_bank}

Step 2:#### If the subject is NOT explicitly in the quiz bank, respond ONLY with:
"I'm sorry I do not have information about that"
DO NOT generate questions about related topics.

Step 3:#### If the subject IS in the quiz bank, identify the category and generate 3 questions.



Use the following format for the quiz:
Question 1:{delimiter} <question 1>

Question 2:{delimiter} <question 2>

Question 3:{delimiter} <question 3>

"""

Use langchain to build the prompt template.

In [10]:
from langchain_core.prompts import ChatPromptTemplate
chat_prompt = ChatPromptTemplate.from_messages([("human", prompt_template)])

In [11]:
# print to observe the content or generated object
chat_prompt

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\nFollow these steps to generate a customized quiz for the user.\nThe question will be delimited with four hashtags i.e ####\n\nThe user will provide a category that they want to create a quiz for. Any questions included in the quiz\nshould only refer to the category.\n\nStep 1:#### First decide whether the user is asking about a subject that is in the following quiz bank:\n- Geography: Paris, Rome (historical), Mediterranean climate\n- Science: Photosynthesis, Newton\'s Laws, Quantum Mechanics  \n- Art: Mona Lisa, Starry Night, Impressionism\n\nHere is the quiz bank\n\n1. Subject: Leonardo DaVinci\n   Categories: Art, Science\n   Facts:\n    - Painted the Mona Lisa\n    - Studied zoology, anatomy, geology, optics\n    - Designed a flying machine\n\n2. Subject: Paris\n   Categories: Ar

Choose the LLM.

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-3.5-turbo",   # or "gpt-4o", "gpt-4o-mini", etc.
    temperature=0
)

# Test it
print(llm)

profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x14028c590> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x14028d010> root_client=<openai.OpenAI object at 0x13f9b9940> root_async_client=<openai.AsyncOpenAI object at 0x14028cd70> temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True


Set up an output parser in LangChain that converts the llm response into a string.

In [13]:
# parser
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser

StrOutputParser()

Connect the pieces using the pipe operator from Langchain Expression Language.

In [14]:
chain = chat_prompt | llm | output_parser
chain

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\nFollow these steps to generate a customized quiz for the user.\nThe question will be delimited with four hashtags i.e ####\n\nThe user will provide a category that they want to create a quiz for. Any questions included in the quiz\nshould only refer to the category.\n\nStep 1:#### First decide whether the user is asking about a subject that is in the following quiz bank:\n- Geography: Paris, Rome (historical), Mediterranean climate\n- Science: Photosynthesis, Newton\'s Laws, Quantum Mechanics  \n- Art: Mona Lisa, Starry Night, Impressionism\n\nHere is the quiz bank\n\n1. Subject: Leonardo DaVinci\n   Categories: Art, Science\n   Facts:\n    - Painted the Mona Lisa\n    - Studied zoology, anatomy, geology, optics\n    - Designed a flying machine\n\n2. Subject: Paris\n   Categories: Ar

Build the function 'assistance_chain' to put together all steps above.

In [15]:
# taking all components and making reusable as one piece
def assistant_chain(
    system_message,
    human_template="{question}",
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    output_parser=StrOutputParser()):
  
  chat_prompt = ChatPromptTemplate.from_messages([
      ("system", system_message),
      ("human", human_template),
  ])
  return chat_prompt | llm | output_parser

### Evaluations

Create the function 'eval_expected_words' for the first example.

In [16]:
def eval_expected_words(
    system_message,
    question,
    expected_words,
    human_template="{question}",
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    output_parser=StrOutputParser()):
    
  assistant = assistant_chain(
      system_message,
      human_template,
      llm,
      output_parser)
    
  
  answer = assistant.invoke({"question": question})
    
  print(answer)
    
  assert any(word in answer.lower() \
             for word in expected_words), \
    f"Expected the assistant questions to include \
    '{expected_words}', but it did not"

Test: Generate a quiz about science.

In [17]:
question  = "Generate a quiz about science."
expected_words = ["davinci", "telescope", "physics", "curie"]

Create the eval.

In [18]:
eval_expected_words(
    prompt_template,
    question,
    expected_words
)

#### First decide whether the user is asking about a subject that is in the following quiz bank:
- Geography: Paris, Rome (historical), Mediterranean climate
- Science: Photosynthesis, Newton's Laws, Quantum Mechanics  
- Art: Mona Lisa, Starry Night, Impressionism

Here is the quiz bank

1. Subject: Leonardo DaVinci
   Categories: Art, Science
   Facts:
    - Painted the Mona Lisa
    - Studied zoology, anatomy, geology, optics
    - Designed a flying machine

2. Subject: Paris
   Categories: Art, Geography
   Facts:
    - Location of the Louvre, the museum where the Mona Lisa is displayed
    - Capital of France
    - Most populous city in France
    - Where Radium and Polonium were discovered by scientists Marie and Pierre Curie

3. Subject: Telescopes
   Category: Science
   Facts:
    - Device to observe different objects
    - The first refracting telescopes were invented in the Netherlands in the 17th Century
    - The James Webb space telescope is the largest telescope in space

Create the function 'evaluate_refusal' to define a failing test case where the app should decline to answer.

In [19]:
def evaluate_refusal(
    system_message,
    question,
    decline_response,
    human_template="{question}", 
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    output_parser=StrOutputParser()):
    
  assistant = assistant_chain(human_template, 
                              system_message,
                              llm,
                              output_parser)
  
  answer = assistant.invoke({"question": question})
  print(answer)
  
  assert decline_response.lower() in answer.lower(), \
    f"Expected the bot to decline with \
    '{decline_response}' got {answer}"

Define a new question (which should be a bad request)

In [20]:
question  = "Generate a quiz about Rome."
decline_response = "I'm sorry"

Create the refusal eval.

<p style="background-color:pink; padding:15px;"> <b>Note:</b> The following function call will throw an exception.</p>


In [21]:
evaluate_refusal(
    prompt_template,
    question,
    decline_response
)

#### Step 1:
I see you are interested in Rome (historical).

#### Step 2:
I'm sorry I do not have information about that.

#### Step 3:


## Running evaluations in a CircleCI pipeline

Put all these steps together into files to reuse later.

**_Note:_** fixing the system_message by adding additional rules:

- Only use explicit matches for the category, if the category is not an exact match to categories in the quiz bank, answer that you do not have information.
- If the user asks a question about a subject you do not have information about in the quiz bank, answer "I'm sorry I do not have information about that".

In [22]:
%%writefile app.py
from langchain_core.prompts           import ChatPromptTemplate
from langchain_openai                 import ChatOpenAI
from langchain_core.output_parsers    import StrOutputParser

delimiter = "####"

quiz_bank = """1. Subject: Leonardo DaVinci
   Categories: Art, Science
   Facts:
    - Painted the Mona Lisa
    - Studied zoology, anatomy, geology, optics
    - Designed a flying machine
  
2. Subject: Paris
   Categories: Art, Geography
   Facts:
    - Location of the Louvre, the museum where the Mona Lisa is displayed
    - Capital of France
    - Most populous city in France
    - Where Radium and Polonium were discovered by scientists Marie and Pierre Curie

3. Subject: Telescopes
   Category: Science
   Facts:
    - Device to observe different objects
    - The first refracting telescopes were invented in the Netherlands in the 17th Century
    - The James Webb space telescope is the largest telescope in space. It uses a gold-berillyum mirror

4. Subject: Starry Night
   Category: Art
   Facts:
    - Painted by Vincent van Gogh in 1889
    - Captures the east-facing view of van Gogh's room in Saint-Rémy-de-Provence

5. Subject: Physics
   Category: Science
   Facts:
    - The sun doesn't change color during sunset.
    - Water slows the speed of light
    - The Eiffel Tower in Paris is taller in the summer than the winter due to expansion of the metal.
"""

system_message = f"""
Follow these steps to generate a customized quiz for the user.
The question will be delimited with four hashtags i.e {delimiter}

The user will provide a category that they want to create a quiz for. Any questions included in the quiz
should only refer to the category.

Step 1:#### First decide whether the user is asking about a subject that is in the following quiz bank:
- Geography: Paris, Rome (historical), Mediterranean climate
- Science: Photosynthesis, Newton's Laws, Quantum Mechanics  
- Art: Mona Lisa, Starry Night, Impressionism

Here is the quiz bank

{quiz_bank}

Step 2:#### If the subject is NOT explicitly in the quiz bank, respond ONLY with:
"I'm sorry I do not have information about that"
DO NOT generate questions about related topics.

Step 3:#### If the subject IS in the quiz bank, identify the category and generate 3 questions.

Use the following format for the quiz:
Question 1:{delimiter} <question 1>

Question 2:{delimiter} <question 2>

Question 3:{delimiter} <question 3>

Additional rules:

- Only use explicit matches for the category, if the category is not an exact match to categories in the quiz bank, answer that you do not have information.
- If the user asks a question about a subject you do not have information about in the quiz bank, answer "I'm sorry I do not have information about that".
"""

"""
  Helper functions for writing the test cases
"""

def assistant_chain(
    system_message=system_message,
    human_template="{question}",
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    output_parser=StrOutputParser()):

  chat_prompt = ChatPromptTemplate.from_messages([
      ("system", system_message),
      ("human", human_template),
  ])
  return chat_prompt | llm | output_parser


Overwriting app.py


Command to see the content:

In [23]:
!cat app.py

from langchain_core.prompts           import ChatPromptTemplate
from langchain_openai                 import ChatOpenAI
from langchain_core.output_parsers    import StrOutputParser

delimiter = "####"

quiz_bank = """1. Subject: Leonardo DaVinci
   Categories: Art, Science
   Facts:
    - Painted the Mona Lisa
    - Studied zoology, anatomy, geology, optics
    - Designed a flying machine

2. Subject: Paris
   Categories: Art, Geography
   Facts:
    - Location of the Louvre, the museum where the Mona Lisa is displayed
    - Capital of France
    - Most populous city in France
    - Where Radium and Polonium were discovered by scientists Marie and Pierre Curie

3. Subject: Telescopes
   Category: Science
   Facts:
    - Device to observe different objects
    - The first refracting telescopes were invented in the Netherlands in the 17th Century
    - The James Webb space telescope is the largest telescope in space. It uses a gold-berillyum mirror

4. Subject: Starry Night
   Category: 

Create new file to include the evals.

In [24]:
%%writefile test_assistant.py
from app import assistant_chain
from app import system_message
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


def eval_expected_words(
    system_message,
    question,
    expected_words,
    human_template="{question}",
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    output_parser=StrOutputParser()):

    assistant = assistant_chain(system_message=system_message)  # ← Use keyword arg
    answer = assistant.invoke({"question": question})
    print(answer)

    assert any(word in answer.lower() for word in expected_words), \
        f"Expected the assistant questions to include '{expected_words}', but it did not"


def evaluate_refusal(
    system_message,
    question,
    decline_response,
    human_template="{question}", 
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    output_parser=StrOutputParser()):

    # FIX: Use keyword arguments for clarity
    assistant = assistant_chain(
        system_message=system_message,
        human_template=human_template,
        llm=llm,
        output_parser=output_parser
    )

    answer = assistant.invoke({"question": question})
    print(answer)

    assert decline_response.lower() in answer.lower(), \
        f"Expected the bot to decline with '{decline_response}' got {answer}"


"""
  Test cases
"""

def test_science_quiz():
    question = "Generate a quiz about science."
    expected_subjects = ["davinci", "telescope", "physics", "curie"]
    eval_expected_words(system_message, question, expected_subjects)


def test_geography_quiz():
    question = "Generate a quiz about geography."
    expected_subjects = ["paris", "france", "louvre"]
    eval_expected_words(system_message, question, expected_subjects)


def test_refusal_rome():
    question = "Help me create a quiz about Rome"
    decline_response = "I'm sorry"
    evaluate_refusal(system_message, question, decline_response)

Overwriting test_assistant.py


Command to see the content of the file:

In [25]:
!cat test_assistant.py

from app import assistant_chain
from app import system_message
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


def eval_expected_words(
    system_message,
    question,
    expected_words,
    human_template="{question}",
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),
    output_parser=StrOutputParser()):

    assistant = assistant_chain(system_message=system_message)  # ← Use keyword arg
    answer = assistant.invoke({"question": question})
    print(answer)

    assert any(word in answer.lower() for word in expected_words), \
        f"Expected the assistant questions to include '{expected_words}', but it did not"


def evaluate_refusal(
    system_message,
    question,
    decline_response,
    human_template="{question}", 
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),


### The CircleCI config file
Now let's set up our tests to run automatically in CircleCI.

For this course, we've created a working CircleCI config file. Let's take a look at the configuration.


In [26]:
!cat circle_config.yml

version: 2.1
orbs:
  # The python orb contains a set of prepackaged circleci configuration you can use repeatedly in your configurations files
  # Orb commands and jobs help you with common scripting around a language/tool
  # so you dont have to copy and paste it everywhere.
  # See the orb documentation here: https://circleci.com/developer/orbs/orb/circleci/python
  python: circleci/python@2.1.1

parameters:
  eval-mode:
    type: string
    default: "commit"

workflows:
  evaluate-commit:
    when:
      equal: [ commit, << pipeline.parameters.eval-mode >> ]
    jobs:
      - run-commit-evals:
          context:
            - dl-ai-courses
  evaluate-release:
    when:
      equal: [ release, << pipeline.parameters.eval-mode >> ]
    jobs:
      - run-pre-release-evals:
          context:
            - dl-ai-courses
  evaluate-all:
    when:
      equal: [ full, << pipeline.parameters.eval-mode >> ]
    jobs:
      - run-manual-evals:
          context:
            - dl-ai-courses
 

## Run the per-commit evals
Push files into the github repo.

CircleCI pipeline is automatically triggers per project setting to "every push".

In [28]:
from utils import push_files, get_branch

# Push all necessary files including requirements
push_files(
    course_repo, 
    course_branch, 
    ["app.py", "test_assistant.py", "requirements.txt"]
)

uploading test_assistant.py
uploading app.py
uploading requirements.txt
pushing files to: dl-cci-best-selling-puck-80
dl-cci-best-selling-puck-80 already exists in the repository pushing updated changes
